In [1]:
#Q.11 요일x시간대 전환 패턴과 광고 시간 최적화
# 포인트: 트래픽만 보지말고 전환율도 함께 관찰하자 -> 퍼포먼스 마케팅

# 1. 세션별 시작 시각(최초 이벤트)으로 요일/시간대 부여
# 2. 시간대별 세션수·전환율 프로파일, 요일x시간대 세션수·전환율 피벗 2종
# 3. 트래픽 상위 시간대 vs 전환율 상위 시간대 대조 -> 차등 입찰 근거 판단
# 4. 제출물: 시간대/요일 프로파일 표, 요일x시간대 피벗 2종, 입찰 조정 의견
# 주의: 새벽 등 세션수(분모) 작은 셀은 전환율이 튈 수 있어 최소 표본 기준으로 걸러야 함

In [2]:
import pandas as pd

path = "../data/web_logs.csv"
usecols = ["session_id", "event_time", "event_type"]
dtype = {"session_id": "string", "event_type": "category"}

logs = pd.read_csv(path, usecols=usecols, dtype=dtype)
logs["event_time"] = pd.to_datetime(logs["event_time"])

# 세션 시작 시각 = 세션별 event_time 최솟값
#같은 session_id에 속한 이벤트 중 가장 이른 시각 -> start_time으로 변경
sess = logs.groupby("session_id")["event_time"].min().rename("start_time").reset_index()

#세션 시작 요일, 시간대 구하기
sess["dow"] = sess["start_time"].dt.dayofweek
sess["hour"] = sess["start_time"].dt.hour

#구매가 발생한 세션 찾기 (set()으로 중복제거)
purchased_sessions = set(logs.loc[logs["event_type"] == "purchase", "session_id"]) #구매가 발생한 행에서 session_id만 가져오기

#astype() boolean 값을 정수로 (1, 0)로 변환
sess["purchased"] = sess["session_id"].isin(purchased_sessions).astype(int)

#sess.shape (전체 행수, 전체 컬럼 수) 245430 세션 수, 5개 column
#세션별 데이터생성
print(sess.shape, "전체 세션 전환율:", round(sess["purchased"].mean(), 4))
sess.head()

(245430, 5) 전체 세션 전환율: 0.2802


,session_id,start_time,dow,hour,purchased
0,sess_000011ca,2024-03-10 07:17:51,6,7,0
1,sess_0000308b,2024-02-06 03:32:56,1,3,1
2,sess_00008045,2024-03-14 18:38:35,3,18,0
3,sess_00008d8b,2024-04-07 19:06:53,6,19,1
4,sess_0000b3ec,2024-02-21 06:37:42,2,6,0


In [3]:
# 시간대별 프로파일 (세션수 · 전환율)

hour_profile = sess.groupby("hour").agg(sessions=("session_id", "count"), conversion=("purchased", "mean")).round(4)
hour_profile

,sessions,conversion
hour,,
0,10353,0.2797
1,10181,0.2830
2,10227,0.2753
3,10456,0.2833
4,10287,0.2812
5,10186,0.2721
6,10276,0.2739
7,10401,0.2806
8,10273,0.2815


In [4]:
sess.info()

<class 'pandas.DataFrame'>
RangeIndex: 245430 entries, 0 to 245429
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   session_id  245430 non-null  string        
 1   start_time  245430 non-null  datetime64[us]
 2   dow         245430 non-null  int32         
 3   hour        245430 non-null  int32         
 4   purchased   245430 non-null  int64         
dtypes: datetime64[us](1), int32(2), int64(1), string(1)
memory usage: 10.6 MB


In [11]:
# 요일x시간대 피벗 2종: 세션수, 전환율
dow_name = {0: "월", 1: "화", 2: "수", 3: "목", 4: "금", 5: "토", 6: "일"}

#dow_name이라는 column을 새로 생성
#.map(dow_name) 을 이용해 숫자를 요일 이름으로 변환한것
sess["dow_name"] =sess["dow"].map(dow_name)

#요일 X 시간대 session 수 pivot
pivot_sessions = sess.pivot_table(
    index='dow_name',
    columns="hour",
    values="session_id",
    aggfunc="count",
    fill_value=0
)

#요일 순서를 월욜부터 일욜까지 다시 정렬
pivot_sessions = pivot_sessions.reindex(["월", "화", "수", "목", "금", "토", "일"])

#요일 X 시간대 전환율 피벗
pivot_conversion = sess.pivot_table(index="dow_name", columns="hour", values="purchased", aggfunc="mean")
pivot_conversion = pivot_conversion.reindex(["월", "화", "수", "목", "금", "토", "일"]).round(4)

#셀당 세션 수 최솟값과 최댓값을 확인해서 세션 고루고루 세션수기 분포 되어있는지 확인
print("셀당 세션수 min/max:", pivot_sessions.values.min(), "/", pivot_sessions.values.max())
pivot_sessions  
# pivot_sessions.info()

셀당 세션수 min/max: 1253 / 1629


hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
dow_name,,,,,,,,,,,,,,,,,,,,,
월,1568,1559,1532,1595,1532,1549,1629,1609,1545,1581,...,1561,1526,1601,1519,1523,1581,1516,1546,1588,1559
화,1544,1453,1534,1534,1573,1552,1585,1510,1562,1532,...,1492,1513,1546,1526,1524,1519,1503,1507,1529,1549
수,1510,1547,1445,1472,1538,1485,1522,1510,1437,1521,...,1503,1490,1441,1535,1507,1569,1486,1420,1531,1455
목,1473,1488,1492,1491,1426,1513,1419,1470,1496,1445,...,1378,1471,1399,1413,1410,1459,1445,1479,1476,1480
금,1421,1400,1490,1506,1459,1331,1413,1451,1459,1467,...,1412,1461,1434,1418,1479,1454,1361,1439,1420,1413
토,1430,1387,1373,1475,1403,1366,1329,1438,1379,1401,...,1384,1386,1443,1417,1385,1434,1313,1444,1446,1327
일,1407,1347,1361,1383,1356,1390,1379,1413,1395,1323,...,1359,1325,1413,1320,1274,1323,1363,1253,1347,1305


In [9]:
sess.info()
sess[["dow", "dow_name"]]

<class 'pandas.DataFrame'>
RangeIndex: 245430 entries, 0 to 245429
Data columns (total 6 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   session_id  245430 non-null  string        
 1   start_time  245430 non-null  datetime64[us]
 2   dow         245430 non-null  int32         
 3   hour        245430 non-null  int32         
 4   purchased   245430 non-null  int64         
 5   dow_name    245430 non-null  str           
dtypes: datetime64[us](1), int32(2), int64(1), str(1), string(1)
memory usage: 13.2 MB


,dow,dow_name
0,6,일
1,1,화
2,3,목
3,6,일
4,2,수
...,...,...
245425,0,월
245426,3,목
245427,5,토
245428,1,화


In [6]:
pivot_conversion

hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
dow_name,,,,,,,,,,,,,,,,,,,,,
월,0.2545,0.2976,0.2898,0.2978,0.2866,0.2744,0.2713,0.2902,0.2816,0.2676,...,0.2671,0.2903,0.2904,0.2910,0.2797,0.2688,0.2764,0.2950,0.2790,0.2784
화,0.2740,0.2870,0.2790,0.2862,0.2619,0.2629,0.2675,0.2834,0.3175,0.3074,...,0.2895,0.3113,0.2904,0.2844,0.3005,0.2778,0.2828,0.2721,0.2937,0.2757
수,0.2901,0.2883,0.2775,0.2826,0.2724,0.2721,0.2792,0.2914,0.2804,0.2847,...,0.2681,0.3020,0.2665,0.3023,0.2986,0.2804,0.2820,0.2718,0.2769,0.2962
목,0.2967,0.2547,0.2849,0.2803,0.2945,0.2690,0.2833,0.2714,0.2741,0.2934,...,0.2823,0.2644,0.2709,0.2682,0.2596,0.2735,0.2782,0.2691,0.2832,0.2872
금,0.2885,0.2771,0.2523,0.2742,0.3050,0.2750,0.2590,0.2715,0.2591,0.2699,...,0.2833,0.2902,0.2936,0.2708,0.2853,0.2572,0.2983,0.2613,0.2528,0.2909
토,0.2853,0.2870,0.2644,0.2685,0.2694,0.2694,0.2897,0.2712,0.2661,0.2784,...,0.2695,0.3009,0.2848,0.2858,0.2751,0.2866,0.2742,0.2777,0.2642,0.2419
일,0.2708,0.2888,0.2777,0.2928,0.2802,0.2835,0.2690,0.2838,0.2889,0.2812,...,0.2818,0.2891,0.2739,0.2530,0.2912,0.2668,0.2605,0.2913,0.2806,0.2881


**핵심 판단 - 최소 표본 기준**: 요일x시간대 셀 중 세션수가 `MIN_SESSIONS`(300) 미만이면 전환율을 신뢰하지 않고 마스킹(NaN) 처리한 뒤에만 상위/하위 비교에 사용한다.

In [7]:
MIN_SESSIONS = 300
stable_conversion = pivot_conversion.where(pivot_sessions >= MIN_SESSIONS)
n_masked = pivot_sessions.size - (pivot_sessions >= MIN_SESSIONS).sum().sum()
print(f"최소표본({MIN_SESSIONS}건) 미달로 마스킹된 셀 수: {n_masked} / {pivot_sessions.size}")

cell_conv = stable_conversion.stack()
cell_sessions = pivot_sessions.stack()

print("\n트래픽(세션수) 상위 5 요일x시간대:")
print(cell_sessions.sort_values(ascending=False).head())
print("\n전환율 상위 5 요일x시간대 (표본 필터 적용):")
print(cell_conv.sort_values(ascending=False).head())
print("\n전체 셀 전환율 평균/표준편차:", round(cell_conv.mean(), 4), "/", round(cell_conv.std(), 4))

최소표본(300건) 미달로 마스킹된 셀 수: 0 / 168

트래픽(세션수) 상위 5 요일x시간대:
dow_name  hour
월         6       1629
          7       1609
          16      1601
          11      1601
          3       1595
dtype: int64

전환율 상위 5 요일x시간대 (표본 필터 적용):
dow_name  hour
화         8       0.3175
          15      0.3113
          9       0.3074
월         13      0.3053
금         4       0.3050
dtype: float64

전체 셀 전환율 평균/표준편차: 0.2801 / 0.0126


### 진단 및 입찰 조정 의견

- 시간대별 세션수는 9,987~10,456건, 요일x시간대 셀은 최소 1,253건으로 이미 `MIN_SESSIONS` 기준을 전부 충족 -> 이 데이터에서는 표본 부족으로 인한 착시는 아니다.
- 그런데도 전환율은 168개 셀 전체가 평균 28.0% ± 1.3%p 안에서만 흔들린다. 이는 세션수 n≈1,450 기준 이항분포 표준오차(약 1.2%p)와 거의 같은 수준 - 즉 요일x시간대 간 전환율 차이는 통계적 잡음과 구분되지 않는다.
- 트래픽 상위 시간대(월요일 06~07시, 16시, 11시 등)와 전환율 상위 시간대(화요일 08·09·15시, 월요일 13시 등)는 겹치지 않지만, 그 차이 자체가 잡음 범위 안이라 "전환 잘 되는 시간에 트래픽이 없다"는 통념과 반대되는 근거로 쓰기도 어렵다.
- **결론**: 이 데이터에는 요일/시간대별 차등 입찰을 정당화할 통계적 근거가 없다. 현재 입찰 전략을 시간대 균등으로 유지하고, 대신 실제 A/B 테스트(예: 특정 요일x시간대 입찰 상향 후 전환율 변화 측정)로 검증한 뒤 조정할 것을 제안한다.